# Drako Edits - Documentación del Proyecto

**Proyecto personal de automatización de contenido para redes sociales (meme reactions, edits, etc.)**

Pipeline general: Descargar material (IG/YT) → Editar/Procesar → Generar video → Subir

Estado actual: Tools de descarga y edición **funcionando**. Generador `meme_reaction` **funcionando**. Automatización end-to-end en desarrollo.

## Estructura de Directorios

```
drako-edits/
├── tools/                    # Herramientas de descarga y edición
│   ├── youtube/              # Descarga de YT (video + audio)
│   │   ├── download_video_yt.py
│   │   └── download_audio_yt.py
│   ├── edit/                 # Edición de video (trim, clean)
│   │   ├── trim_video.py
│   │   └── clean_video.py
│   └── instagram/            # Descarga de IG (posts, singles)
│       ├── __init__.py
│       ├── ig_tracker.py     # Control de límites diarios
│       ├── posts_nologin.py  # Batch de fotos sin login
│       ├── posts_login.py    # Batch de fotos con login
│       ├── single_nologin.py # Post individual sin login
│       └── single_login.py   # Post individual con login
├── tools_output/             # Material descargado
│   ├── posts/                # Fotos de IG (subcarpetas por user)
│   ├── videos/               # Videos de YT
│   └── audios/               # Audios de YT
├── generators/               # Generadores de contenido final
│   └── meme_reaction.py     # Genera video meme+clip+caption
├── automatizaciones/         # Pipelines automatizados end-to-end
│   └── auto_meme_reaction.py
├── assets/                   # Configs JSON por tipo de video
│   ├── meme_reaction/configs/
│   ├── generic/configs/
│   ├── super_freaky_girl/
│   ├── si_no_te_quieres_banar/
│   └── video_generic/configs/
├── output/                   # Videos generados listos para subir
│   ├── meme_reaction/
│   └── si_no_te_quieres_banar/
├── Pending/                  # Scripts pendientes de actualizar
├── obsoletos/                # Scripts deprecados (no usar)
├── .env                      # Credenciales (IG, OpenAI)
└── requirements.txt
```

## Dependencias (`requirements.txt`)

| Paquete | Versión | Uso |
| --- | --- | --- |
| moviepy | >=2.0.0 | Composición de video (generators) |
| Pillow | >=9.2.0 | Manipulación de imágenes (captions, resize) |
| numpy | >=1.24.0 | Detección de barras negras |
| openai | >=1.0.0 | Clasificación de memes con Vision API |
| python-dotenv | >=1.0.0 | Cargar .env con credenciales |
| instagrapi | >=2.0.0 | API de Instagram (auto_meme_reaction) |
| google-api-python-client | >=2.100.0 | Upload a YouTube |
| google-auth-oauthlib | >=1.1.0 | Auth OAuth para YouTube |
| yt-dlp | >=2024.1.0 | Descarga de YouTube (video/audio) |
| requests | >=2.28.0 | Descargas HTTP generales |

**Nota**: `instaloader` se usa en `tools/instagram/` (no está en requirements, se instala aparte).

## `tools/youtube/download_video_yt.py`

**Descargador de video de YouTube** (yt-dlp)

**Funcionalidad:**
- Descarga video de cualquier URL de YouTube (shorts, normales, etc.)
- 2 modos: con audio `(audio).mp4` o sin audio `(sinaudio).mp4`
- Si eliges sin audio, opción de descargar el audio aparte como MP3

**Output:** `tools_output/videos/`

**Uso:**
```bash
# Interactivo
python tools/youtube/download_video_yt.py

# Directo
python tools/youtube/download_video_yt.py --url "URL" --name "clip" --mode audio
python tools/youtube/download_video_yt.py --url "URL" --name "clip" --mode sinaudio --also-audio --audio-name "beat"
```

**Convención de nombres:**
- `{nombre} (audio).mp4` — video con audio incluido
- `{nombre} (sinaudio).mp4` — video sin audio (para usar audio externo)

## `tools/youtube/download_audio_yt.py`

**Descargador de audio de YouTube** (yt-dlp)

**Funcionalidad:**
- Descarga solo audio (MP3) de cualquier URL de YouTube
- Calidad máxima (`--audio-quality 0`)

**Output:** `tools_output/audios/`

**Uso:**
```bash
python tools/youtube/download_audio_yt.py
python tools/youtube/download_audio_yt.py --url "URL" --name "beat_epico"
```

## `tools/edit/trim_video.py`

**Recortador de video** (ffmpeg)

**Funcionalidad:**
- Lista videos en `tools_output/videos/` con duración
- Pides tiempo de inicio y fin (acepta formatos: `30`, `1:30`, `1:05:30`)
- Resultado se guarda en la misma carpeta con sufijo `(trim)`
- Fix especial para clips cortos (<2s): fuerza re-encode con ultrafast para evitar archivos de 0KB
- Para clips normales: intenta stream copy (rápido), fallback a re-encode si hay imprecisión

**Output:** `tools_output/videos/{nombre} (trim).mp4`

**Uso:**
```bash
python tools/edit/trim_video.py
```

## `tools/edit/clean_video.py`

**Limpiador de barras negras** (ffmpeg cropdetect)

**Funcionalidad:**
- Detecta barras negras (letterboxing) automáticamente usando `ffmpeg cropdetect`
- Analiza múltiples puntos del video (20%, 40%, 60% de duración)
- Muestra cuántos píxeles arriba/abajo va a remover
- Exporta video limpio con re-encode (libx264, CRF 18)

**Configuración:**
- `BLACK_THRESHOLD = 20` — luminosidad máxima para considerar "negro"
- `MIN_BAR_PIXELS = 10` — mínimo de píxeles para considerarlo barra

**Output:** `tools_output/videos/{nombre} (clean).mp4`

**Uso:**
```bash
python tools/edit/clean_video.py
```

## `tools/instagram/` — Descargadores de Instagram

Módulo completo con control de rate-limiting y warm-up progresivo.

**Arquitectura:**
- `ig_tracker.py` — Core de control de límites (compartido por todos los scripts)
- `posts_nologin.py` — Batch de fotos de un perfil público (sin login)
- `posts_login.py` — Batch de fotos con cuenta burner (más capacidad)
- `single_nologin.py` — Post individual por URL (sin login)
- `single_login.py` — Post individual por URL (con login)

**Librería usada:** `instaloader` (no `instagrapi` — esa se usa en automatizaciones)

**Output:** `tools_output/posts/{username}/` (batch) o `tools_output/posts/_single/` (individual)

## `tools/instagram/ig_tracker.py`

**Control de límites diarios con warm-up progresivo**

**Funcionalidad:**
- Trackea requests por día, por método (nologin/login), por cuenta
- Warm-up progresivo para evitar bans:

| Día | Sin login | Con login |
| --- | --- | --- |
| 1 | 20 | 50 |
| 2 | 40 | 100 |
| 3 | 60 | 200 |
| 4+ | 100 | 300 |

**Archivo:** `tools/instagram/ig_usage_log.json`

**Funciones públicas:**
- `get_daily_limit(method)` — Límite de hoy según warm-up
- `get_today_usage(method, account)` — Requests usados hoy
- `get_remaining(method, account)` — Cuántos quedan
- `log_request(method, account, shortcode, media_type, username)` — Registrar uso
- `show_status(method, account)` — Muestra dashboard de estado
- `check_can_download(method, account, count)` — Verifica si se puede descargar

## `tools/instagram/posts_nologin.py`

**Descarga batch de fotos de un perfil público (SIN LOGIN)**

**Funcionalidad:**
- Escanea perfil y muestra estadísticas (fotos/videos/carousels)
- Descarga solo fotos (GraphImage + fotos de carousels)
- Respeta límites de ig_tracker
- Delays conservadores: 5s entre posts, pausa 3min cada 15 descargas

**Uso:**
```bash
python tools/instagram/posts_nologin.py
python tools/instagram/posts_nologin.py --username "cuenta" --max 20
python tools/instagram/posts_nologin.py --status
```

## `tools/instagram/posts_login.py`

**Descarga batch de fotos con cuenta burner (CON LOGIN)**

**Funcionalidad:**
- Igual que `posts_nologin.py` pero con login
- Mayor capacidad (300/día vs 100)
- Puede acceder a perfiles privados que la burner siga
- Delays menos conservadores: 3s entre posts, pausa 2min cada 30

**Configuración:** Editar `BURNER_ACCOUNT` directamente en el archivo.

**Uso:**
```bash
python tools/instagram/posts_login.py
python tools/instagram/posts_login.py --username "cuenta" --max 50
```

## `tools/instagram/single_nologin.py`

**Descarga post individual por URL (SIN LOGIN)**

**Funcionalidad:**
- Acepta URLs de posts (`/p/`), reels (`/reel/`), y TV (`/tv/`)
- Descarga fotos Y videos (a diferencia del batch que solo fotos)
- Muestra info del post antes de descargar (tipo, owner, fecha, caption)
- Registra en ig_tracker

**Output:** `tools_output/posts/_single/`

**Uso:**
```bash
python tools/instagram/single_nologin.py
python tools/instagram/single_nologin.py --url "https://instagram.com/p/XXXXX/"
python tools/instagram/single_nologin.py --status
```

## `tools/instagram/single_login.py`

**Descarga post individual por URL (CON LOGIN)**

**Funcionalidad:**
- Igual que `single_nologin.py` pero con cuenta burner
- Mayor capacidad (300/día)
- Configurar `BURNER_ACCOUNT` en el archivo

**Uso:**
```bash
python tools/instagram/single_login.py
python tools/instagram/single_login.py --url "URL"
```

## `generators/meme_reaction.py`

**Generador de videos "Meme Reaction"** — El generator principal y funcional

**Formato del video:**
- 1080x1920 (vertical, para Reels/Shorts/TikTok)
- Imagen (meme) arriba + Video clip (reacción) abajo
- Sin crop: ambos se muestran completos, fondo blanco si sobra espacio
- Split dinámico: meme siempre ocupa entre 65%-75% del alto
- Caption opcional superpuesto en la frontera meme/video
- Auto-detecta y remueve barras negras del clip

**Audio:**
- Opción 1: Audio del clip (si tiene)
- Opción 2: Audio externo (de `tools_output/audios/`)
- Si clip tiene `(sinaudio)` en el nombre → fuerza selección de externo
- Nunca mezcla ambos

**Navegación:**
- Usa `browse_folder()` genérico para elegir archivos por carpetas
- Permite drill-down en subcarpetas y volver con `0`

**2 Modos:**
1. **Desde JSON config** — Elige un config existente y genera
2. **Manual** — Elige meme, clip, audio, caption interactivamente → opción de guardar como JSON

**Fuentes de material:**
- Imágenes: `tools_output/posts/`
- Videos: `tools_output/videos/`
- Audios: `tools_output/audios/`

**Configs:** `assets/meme_reaction/configs/*.json`

**Output:** `output/meme_reaction/`

**Uso:**
```bash
python generators/meme_reaction.py
python generators/meme_reaction.py --config "assets/meme_reaction/configs/meme01.json"
```

## `automatizaciones/Meme_Reaction/` — Proyecto de Automatización

**Pipeline automatizado end-to-end (EN DESARROLLO)**

Este es un sub-proyecto completo con su propia documentación:
→ Ver: `automatizaciones/Meme_Reaction/DOCUMENTACION`

**Arquitectura:**
- Selenium para scraping de links de posts (NO login)
- instaloader sin login para descarga
- OpenAI Vision para clasificación y verificación
- Generator `meme_reaction.py` para composición de video
- Audio: siempre del clip en automatización
- JSON config guardado para replicar manualmente

**Nota:** El archivo `auto_meme_reaction.py` anterior (en raíz de automatizaciones/) está OBSOLETO. Todo se mueve a `automatizaciones/Meme_Reaction/`.

## `Pending/` — Scripts Pendientes de Actualizar

Estos scripts necesitan actualizarse con los patrones establecidos (browse_folder, audio pattern, etc.):

| Script | Descripción |
| --- | --- |
| `generate_generic.py` | Generador genérico (imagen + audio/video) |
| `generate_video_generic.py` | Similar pero con video base |
| `generate_super_freaky_girl.py` | Edit tipo SFG (intro + letras + secuencias) |
| `generate_sfg_personaje.py` | Variante SFG con personaje |
| `generate_si_no_te_quieres.py` | Edit "si no te quieres bañar" |
| `upload_to_youtube.py` | Upload genérico a YouTube (OAuth) |
| `upload_sfg_to_youtube.py` | Upload SFG a YouTube |

**No modificar hasta que se muevan a su ubicación final y se apliquen los patrones.**

## Patrones y Convenciones

### Path resolution
- `generators/*.py` → `Path(__file__).parent.parent` (sube 1 nivel a raíz)
- `tools/youtube/*.py` → `Path(__file__).parent.parent.parent` (sube 3 niveles)
- `tools/edit/*.py` → Mismo (3 niveles)
- `tools/instagram/*.py` → Mismo (3 niveles)
- `automatizaciones/*.py` → `Path(__file__).parent.parent` (sube 1 nivel)

### Navegación por carpetas (`browse_folder`)
- Función genérica en `meme_reaction.py` (aplicar a todos los generators)
- Muestra carpetas primero con `[>] nombre/ (N archivos)`, luego archivos
- Drill-down con números, volver con `0`
- No sube más arriba del root_dir

### Patrón de audio
- Pregunta: audio del clip o audio externo (nunca ambos)
- Detecta `(sinaudio)` en nombre → fuerza selección de externo
- Si no hay audios y clip es sin audio → avisa que será mudo

### Encoding fix (Windows)
Todos los scripts incluyen al inicio:
```python
if sys.platform == "win32":
    sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')
    sys.stdin = io.TextIOWrapper(sys.stdin.buffer, encoding='utf-8', errors='replace')
```

### Flujo interactivo
- Todos los tools tienen modo interactivo (sin args) + modo directo (con --url, --name, etc.)
- Banner con `"=" * 60` al inicio
- Pasos numerados: `--- PASO N: ---`
- Preguntas recursivas: "Descargar otro? (s/n)"

### Instagram: 2 librerías
- `instaloader` — Usado en `tools/instagram/` (más conservador, mejor para batch)
- `instagrapi` — Usado en `automatizaciones/` (más programático, mejor para automatización)